# Phase 3: Model Training & Evaluation

Train 2 models (Naive Bayes, Logistic Regression) with TF-IDF, compare on test set.

In [ ]:
from pathlib import Path
import pickle
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

DATA_DIR = Path('../data')
MODEL_DIR = Path('../models')

train = pd.read_parquet(DATA_DIR / 'train_clean.parquet')
val = pd.read_parquet(DATA_DIR / 'val_clean.parquet')
test = pd.read_parquet(DATA_DIR / 'test_clean.parquet')
print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

In [ ]:
with open(MODEL_DIR / 'tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)
with open(MODEL_DIR / 'nb_model.pkl', 'rb') as f:
    nb = pickle.load(f)
with open(MODEL_DIR / 'lr_model.pkl', 'rb') as f:
    lr = pickle.load(f)

text_col = 'text_clean' if 'text_clean' in test.columns else 'clean_text'
X_test = tfidf.transform(test[text_col])
y_test = test['sentiment']

In [ ]:
labels = ['negative', 'neutral', 'positive']
nb_pred = nb.predict(X_test)
lr_pred = lr.predict(X_test)

print('=== Naive Bayes (Test) ===')
print(classification_report(y_test, nb_pred, labels=labels, zero_division=0))
print('=== Logistic Regression (Test) ===')
print(classification_report(y_test, lr_pred, labels=labels, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, pred) in zip(axes, [('NB', nb_pred), ('LR', lr_pred)]):
    cm = confusion_matrix(y_test, pred, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(f'{name} — Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## Result Summary

| Model | Accuracy | F1 Macro | F1 Weighted |
|:---|:---:|:---:|:---:|
| Naive Bayes | 82.1% | 61.2% | 81.3% |
| Logistic Regression | 81.1% | **66.7%** | 82.8% |

LR selected for deployment (higher F1 Macro, better neutral class recall).